# 08 · Results & conclusions — the outcome-contingent skeleton

**Execution status:** 🟢 EXECUTED (CPU, outputs saved)
**Plan reference:** PLAN Part I (RQ outcomes) · PREREG §8 (null rule) · Part V (paper)

The study's conclusion, written *before* the real data so no result can be reverse-engineered into a story. For each research question we restate the frozen decision rule, show the synthetic-dry-run recovery as a **placeholder** for the real result, and write out the interpretation the paper will carry **under every possible outcome** — including the honest-null branch. When the GPU phase lands, the placeholders are swapped for real numbers and the matching branch is kept; nothing else changes.

| | |
|---|---|
| **Inputs** | the synthetic recovery from notebooks 05, 06a, 07a (re-run compactly here) |
| **Outputs** | master decision table + per-RQ outcome branches + limitations |
| **Runtime** | ~20 s |

*Project 19 — Anatomy of a Design Skill. Governance: `CLAUDE.md`. Plan: `PLAN.md`. Derivations:
`THEORY.md`. Freeze: `PREREGISTRATION.md`. This notebook imports tested machinery from the `p19`
package and carries the narrative; it never re-implements logic that lives in `src/p19/`.*

In [1]:
%matplotlib inline
# Bootstrap: locate the repo root (repo-relative — no hardcoded paths) and make p19 importable.
import sys, pathlib
_here = pathlib.Path.cwd()
_root = next((c for c in [_here, *_here.parents]
              if (c / "pyproject.toml").exists() and (c / "src" / "p19").exists()), _here)
if str(_root / "src") not in sys.path:
    sys.path.insert(0, str(_root / "src"))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from p19 import REPO_ROOT
np.random.seed(0)                      # notebook-level seed; every generator also takes an explicit seed
pd.set_option("display.max_columns", 40); pd.set_option("display.width", 120)
print("p19 ready · repo:", REPO_ROOT.name)

p19 ready · repo: steering-llm-aesthetics


## How to read this notebook

Every claim in this study must move **both** oracle signals — the objective **POC** (deterministic,
rendered-DOM) and a reliability- and power-gated **Bradley–Terry** preference utility. The
**preregistered null rule** governs the verdict:

> *An ablated component's effect, or a steering direction's effect, is reported as **null** unless it
> moves an objective-metric endpoint OR a preference delta that passes both the reliability and power
> gates. Aesthetic claims not backed by the objective half are inadmissible.* (PREREG §8)

Below, the **"synthetic result"** column is a *placeholder with known ground truth* — it proves the
pipeline recovers what was planted. The **branch** column is the paper's conclusion *if the real result
matches that pattern*. On real data, exactly one branch per RQ is kept.

> ## ⚠️ SYNTHETIC PIPELINE VALIDATION
> The numbers below are **simulated with a known planted ground truth** to prove the analysis
> pipeline recovers it. The master table's numbers come from re-running the Stage-1 and Stage-2 synthetic dry-runs compactly; they demonstrate the decision machinery, not the real finding. **Real data replaces this after the GPU phase (see `RUNBOOK.md`).**
> Every synthetic artifact in this notebook is generated by `p19.synthetic` (unit-tested against the
> real pipeline in `tests/test_synthetic.py`).

In [2]:
# Re-run the synthetic pipelines compactly to populate the master decision table (from 05 + 07a).
from p19 import synthetic as syn, poc, mixedlm, multiplicity, steering, bt_model
# --- Stage 1 ---
df, truth = syn.stage1_dataset(n_prompts=24, seed=0)
res = poc.compute_poc(df); df["poc"] = res["poc"].values
df = df.dropna(subset=["poc"]).reset_index(drop=True)
nec = {f: mixedlm.paired_contrast(df, "FULL", f"LOO-{f}", "poc") for f in syn.FACTORS}
suf = {f: mixedlm.paired_contrast(df, f"AOI-{f}", "NEUTRAL", "poc") for f in syn.FACTORS}
fn = mixedlm.paired_contrast(df, "FULL", "NEUTRAL", "poc")
fb = mixedlm.paired_contrast(df, "FULL", "BEAUTY1", "poc")
pv = {"FULL-NEUTRAL": fn.pvalue, "FULL-BEAUTY1": fb.pvalue}
for f in syn.FACTORS:
    pv[f"FULL-LOO-{f}"] = nec[f].pvalue; pv[f"AOI-{f}-NEUTRAL"] = suf[f].pvalue
holm = {t.name: t.reject for t in multiplicity.confirmatory_holm(pv)}
fac = mixedlm.fit_factorial(syn.add_factor_columns(df), "poc")
pairs = [(a,b) for a in syn.FACTORS for b in syn.FACTORS if a < b]
bh = multiplicity.benjamini_hochberg([fac["coefs"].get(f"{a}:{b}",{}).get("p",1) for a,b in pairs],
                                     [f"{a}×{b}" for a,b in pairs], q=0.10)
nec_rank = " > ".join(sorted(syn.FACTORS, key=lambda f: -nec[f].estimate))
# --- Stage 2 ---
arms, at = syn.stage2_arms(seed=0)
obj = mixedlm.paired_contrast(arms[arms.arm.isin(["steered","unsteered"])].rename(columns={"arm":"cell_id"}),
                              "steered","unsteered","poc")
piv = arms[arms.arm.isin(["steered","unsteered"])].pivot_table(index=["prompt_id","seed"],columns="arm",values="poc")
overall_anti = float((piv["steered"]<piv["unsteered"]).mean())
cos_mat, sig_grid, _, _ = syn.stage2_correspondence(seed=0)
matches = sum(sig_grid[i].argmax()==i for i in range(5))

master = pd.DataFrame([
 ["H-skill","FULL≻NEUTRAL both signals", f"δ_POC={fn.estimate:+.2f}, Holm-sig", "✅ works"],
 ["RQ1","rank necessity; Holm", f"{nec_rank}; C4 null(p={nec['C4'].pvalue:.2f})", "necessary (ranked)"],
 ["RQ2","distributed if none ≥30% gap", f"max suff={max(s.estimate for s in suf.values()):.2f}<{0.30*fn.estimate:.2f}", "distributed"],
 ["RQ3","BH over 10 2FIs", f"survivors {[t.name for t in bh if t.reject]}", "interacting"],
 ["RQ4","FULL>BEAUTY1>NEUTRAL", f"FULL−BEAUTY1={fb.estimate:+.2f} sig", "content"],
 ["RQ5","band AUC≥.80 & sel≥.15 & patch≥25%", "band ≈ 11–20 recovered (06a)", "localized"],
 ["RQ6","§7 four conjuncts", f"all 4 hold; steered−unsteered p={obj.pvalue:.1e}", "sufficient & necessary"],
 ["RQ7","unseen δ>0 & anti-steer<50%", f"overall anti={overall_anti:.2f}; admin-table brittle", "partial"],
 ["RQ8","mean|cos|<.3 & ≥3/5 match", f"{matches}/5 signatures match", "found"],
], columns=["RQ","decision rule (frozen)","synthetic result (placeholder)","branch (if real matches)"]).set_index("RQ")
master

,decision rule (frozen),synthetic result (placeholder),branch (if real matches)
RQ,,,
H-skill,FULL≻NEUTRAL both signals,"δ_POC=+1.90, Holm-sig",✅ works
RQ1,rank necessity; Holm,C5 > C1 > C2 > C3 > C4; C4 null(p=0.62),necessary (ranked)
RQ2,distributed if none ≥30% gap,max suff=0.55<0.57,distributed
RQ3,BH over 10 2FIs,"survivors ['C1×C5', 'C2×C3', 'C3×C5']",interacting
RQ4,FULL>BEAUTY1>NEUTRAL,FULL−BEAUTY1=+1.03 sig,content
RQ5,band AUC≥.80 & sel≥.15 & patch≥25%,band ≈ 11–20 recovered (06a),localized
RQ6,§7 four conjuncts,all 4 hold; steered−unsteered p=1.2e-10,sufficient & necessary
RQ7,unseen δ>0 & anti-steer<50%,overall anti=0.22; admin-table brittle,partial
RQ8,mean|cos|<.3 & ≥3/5 match,5/5 signatures match,found


The pipeline recovers **every planted structure** and renders the corresponding preregistered branch —
the guarantee that the machinery is sound. What follows is the paper's conclusion under *each* possible
real outcome, RQ by RQ.

## H-skill — does the skill work, and is it content?

- **works** *(synthetic branch)* — FULL ≻ NEUTRAL on both signals. Because NEUTRAL is length-matched,
  the gain is **design content, not prompt mass**. The paper leads with the effect size and the
  %-gap FULL closes over NOSYS.
- **null** — FULL ≈ NEUTRAL. The model cannot use a structured design skill at all (would have been
  caught at the pilot, escalating to the ADR-001 fallback). Reported honestly; the rest of Stage 1 is
  scoped to "no measurable skill effect on this model."

## RQ1 — component necessity

- **necessary (ranked)** *(synthetic)* — one or more components, removed, degrade quality; ranked by δ
  (here C5 negatives and C1 color largest). Paper: "the skill's weight concentrates in *telling the
  model what not to do* and *pinning a palette*."
- **null** — no LOO degrades quality (tight CIs at 0). Paper: components are individually dispensable;
  the skill works only as a whole.
- **negative** — removing a component *improves* quality (an over-constraining instruction). Reported
  honestly as a design-guidance lesson.

## RQ2 — component sufficiency

- **distributed** *(synthetic)* — every AOI-Cᵢ lifts above NEUTRAL but none reaches 30 % of the
  FULL−NEUTRAL gap. Paper: sufficiency is *distributed* — no single instruction carries the skill.
- **one-carries** — a single component reaches near-FULL alone. Paper: the skill is essentially that
  one instruction; the rest is polish.
- **null** — no component alone beats NEUTRAL. Paper: components are only effective in combination
  (pairs with the RQ3 interaction story).

## RQ3 — interactions / additivity

- **interacting** *(synthetic)* — named 2FIs survive BH (here C1×C5 super-additive, C2×C3 sub-additive).
  Paper: "negatives help more once a palette is specified" — the components are not independent knobs.
- **additive** — no 2FI survives. Paper: component effects simply add; the skill is a checklist.
- **dominated by a single main effect** — one component's main effect swamps all interactions. Paper:
  the skill is effectively one lever.

## RQ4 — do five words match 391 tokens?

- **content** *(synthetic)* — FULL > BEAUTY1 > NEUTRAL with FULL−BEAUTY1 > 0. Paper: structured
  guidance is load-bearing; a nudge is not a substitute.
- **nudge** — BEAUTY1 reaches ≥ 70 % of the gap and FULL−BEAUTY1 is null. Paper: "make it beautiful"
  captures most of the effect — a striking, cheap result.
- **neither** — BEAUTY1 ≈ NEUTRAL. Paper: the model needs *specific* guidance or none helps.

## RQ5 — where does the effect live?

- **localized band** *(synthetic)* — a contiguous mid-band (≈ 11–20) clears probe-AUC, selectivity,
  and patching. Paper: the design signal is linearly readable and patch-localizable mid-depth; feeds
  the steering focus layers.
- **diffuse / no clean peak** — no band meets all three thresholds. Paper: the signal is distributed;
  steering may need multi-layer injection (honest report, pre-stated fallback).

## RQ6 — the bar: does a no-prompt direction causally reproduce the gain?

- **causally sufficient (and/or necessary)** *(synthetic)* — all four conjuncts hold: steered ≻
  unsteered on ≥1 objective family after Holm, gated BT CI > 0, the norm-matched random control fails,
  and the flip test attenuates. Paper's headline: **a clean-design direction exists and steers**;
  report %-skill-reproduced with CI. If addition succeeds but the flip does not → "**sufficient, not
  demonstrably necessary**."
- **null** — a correlational direction that does not steer. Paper (equally publishable per the brief):
  **design behavior is not linearly steerable** in this model — a legitimate negative result, written
  with the tight-null effect sizes that distinguish it from mere under-power.

## RQ7 — failure surface & generalization

- **partial** *(synthetic)* — helps on average, generalizes to unseen types, but with a named brittle
  regime (here admin-data-tables cross the 50 % anti-steerable line). Paper: the honest failure map —
  steering is not a blanket win.
- **robust** — anti-steerable fraction low across all types. Paper: the direction transfers cleanly OOD.
- **brittle** — OOD collapse (anti-steerable ≥ 50 % overall). Paper: the direction is dev-specific;
  scope the claim to the steerable regime.

## RQ8 — component ↔ direction correspondence (stretch)

- **found** *(synthetic)* — sub-vectors near-orthogonal and ≥ 3/5 steer their own metric family. Paper's
  strongest result: the design skill has a *component basis* inside the model, mirroring the Stage-1
  ablation signatures.
- **partial** — 1–2 signatures match. Paper: some structure, some entanglement.
- **absent** — one entangled direction, no component basis. Paper: still informative — design is a
  single axis, not five.

## Cross-cutting honest-null policy

Any RQ resolving to null is written with **(a)** the effect-size estimate and CI showing the null is
*tight* (the power gate distinguishes this from under-power), **(b)** the objective-signal evidence,
and **(c)** the interpretation. A clean null — a component that does nothing, a direction that does not
steer — is a first-class, reported finding (CLAUDE.md, BRIEF). The synthetic **C4 null** (notebook 05)
and the RQ6 null branch above are the templates.

## Limitations (stated up front)

- **One model.** Every result is about `Qwen2.5-Coder-7B-Instruct`. The two stages describe the *same*
  system by design, but generalization to other models is untested (a deliberate scope choice, ADR-001).
- **40 dev prompts / 18 held-out.** Cluster-bootstrap CIs are calibrated to the *number of prompts*, not
  generations (THEORY §T3.5) — tails are limited at 40 clusters; we report parametric MixedLM CIs as a
  robustness pair and do not claim generalization beyond the 58-brief task population.
- **Judge, not ground truth.** The preference signal is a VLM judge validated on a 100-pair human
  subset; it is gated, but it is not a human panel. The objective half is the unfoolable anchor.
- **Diff-in-means is a linear probe of a nonlinear model.** A null steering result bounds *linear*
  steerability, not the existence of the concept (THEORY §T9.3).
- **Synthetic dry-runs validate the pipeline, not the phenomenon.** Every executed number in this series
  is simulated; the real findings await the GPU phase.

## What the paper claims, and where the real numbers come from

The two-part paper (`paper/`) has outcome-contingent templates matching the branches above: Part 1
(Stage-1 ablation, RQ1–RQ4) and Part 2 (Stage-2 interpretability, RQ5–RQ8). Figures **F1–F13** and
tables **T1–T10** are generated by `p19.figures` from the results tables — no hand-editing.

**Current status.** The entire CPU-safe stack is complete and green (134 tests); the two-signal oracle
is built and validated; the full analysis pipeline recovers every planted effect end-to-end. The GPU
phase — pilot → generation → render/metrics → judging → extraction → sweep/freeze → held-out verify →
stretch — runs on Colab session-by-session per **`RUNBOOK.md`**. When it lands, the placeholders in this
notebook's master table become real numbers and exactly one branch per RQ is kept.

---
*End of the notebook series. Start again at `00_overview.ipynb` for the map, or open `RUNBOOK.md` for
the GPU execution plan.*